# 🟣 SCD Implementation – Players (Type 1 & Type 2)

## 📌 Objective

Handle changes in player data over time using Slowly Changing Dimensions.

## 🧠 Types Implemented

* SCD Type 1 → Overwrite
* SCD Type 2 → History tracking

## 🔁 Type 1

* Updates existing records
* No history maintained

## 🔁 Type 2

* Tracks historical changes
* Adds:

  * start_date
  * end_date
  * is_current flag

## ⚙️ Logic

* Detect changes in key attributes
* Expire old records
* Insert new version

## 📥 Source

* Bronze Players

## 📤 Target

* Silver Players (SCD Enabled)

## 🎯 Goal

Enable historical tracking for analytics


In [0]:
source_df = spark.read.table('game_analytics.bronze.players')
display(source_df)

In [0]:
%sql
MERGE INTO game_analytics.silver.players AS target
USING game_analytics.bronze.players AS source
ON target.player_id = source.player_id

WHEN MATCHED THEN
UPDATE SET
  target.first_name = source.first_name,
  target.last_name  = source.last_name,
  target.level      = CAST(source.level AS INT),
  target.country    = source.country,
  target.created_at = source.created_at

WHEN NOT MATCHED THEN
INSERT (
  player_id,
  first_name,
  last_name,
  joined_date,
  level,
  country,
  created_at,
  ingestion_time
)
VALUES (
  source.player_id,
  source.first_name,
  source.last_name,
  source.joined_data,
  CAST(source.level AS INT),
  source.country,
  source.created_at,
  source.ingestion_time
)